Verificando se o pyspark está instalado corretamente no vsvoce

In [ ]:
import sys
print(sys.executable)


c:\Users\00157NLUC-BrenoR\pos_data_analytics\.venv\Scripts\python.exe


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("SparkLocal").getOrCreate()
spark.range(5).show()


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [ ]:
df = spark.sql("""select 'Sucesso total, estamos online!' as hello""")
df.show()

+--------------------+
|               hello|
+--------------------+
|Sucesso total, es...|
+--------------------+



In [ ]:
# Import spark libraries
from pyspark.sql import Row, DataFrame
from pyspark.sql.types import StringType, StructType, StructField, IntegerType
from pyspark.sql.functions import col, expr, lit, substring, concat, concat_ws, when, coalesce
from pyspark.sql import functions as F  # for more sql functions
from functools import reduce
from pyspark.sql.functions import to_date
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.types import StringType, StructType, StructField

# Data Manipulation using Spark

In [ ]:
caminho = r"C:\Users\00157NLUC-BrenoR\pos_data_analytics\fase_3_big_data\banklist.csv"


In [ ]:
schema = StructType([
    StructField("Bank Name�", StringType(), True),
    StructField("City�", StringType(), True),
    StructField("State�", StringType(), True),
    StructField("Cert�", StringType(), True),
    StructField("Acquiring Institution�", StringType(), True),
    StructField("Closing Date�", StringType(), True),
    StructField("Fund", StringType(), True),
])

In [ ]:
df = spark.read.csv(
    caminho,
    header=True,
    schema=schema,
    encoding="UTF-8"
)


In [ ]:
# Renomeia as colunas
df_clean = (
    df
    .withColumnRenamed("Bank Name�", "bank_name")
    .withColumnRenamed("City�", "city")
    .withColumnRenamed("State�", "state")
    .withColumnRenamed("Cert�", "cert")
    .withColumnRenamed("Acquiring Institution�", "acquiring_institution")
    .withColumnRenamed("Closing Date�", "closing_date")
    .withColumnRenamed("Fund", "fund")
)

In [ ]:
# Remove o caractere � do conteúdo
for c in df_clean.columns:
    df_clean = df_clean.withColumn(c, regexp_replace(col(c), "�", ""))

df_clean.show(5)
df_clean.printSchema()

+--------------------+------------+-----+-----+---------------------+------------+-----+
|           bank_name|        city|state| cert|acquiring_institution|closing_date| fund|
+--------------------+------------+-----+-----+---------------------+------------+-----+
|The Santa Anna Na...|  Santa Anna|   TX| 5520| Coleman County St...|   27-Jun-25|10549|
|Pulaski Savings Bank|     Chicago|   IL|28611|      Millennium Bank|   17-Jan-25|10548|
|First National Ba...|     Lindsay|   OK| 4134| First Bank & Trus...|   18-Oct-24|10547|
|Republic First Ba...|Philadelphia|   PA|27332| Fulton Bank, Nati...|   26-Apr-24|10546|
|       Citizens Bank|    Sac City|   IA| 8758| Iowa Trust & Savi...|    3-Nov-23|10545|
+--------------------+------------+-----+-----+---------------------+------------+-----+
only showing top 5 rows

root
 |-- bank_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- cert: string (nullable = true)
 |-- acquiring_inst

In [ ]:
output_path = r"C:\Users\00157NLUC-BrenoR\pos_data_analytics\fase_3_big_data\banklist_clean"
df_clean.write.mode("overwrite").option("header", True).csv(output_path)

In [ ]:
df_clean.coalesce(1).write.mode("overwrite").option("header", True).csv(output_path)

## Using SQL in PySpark

In [ ]:
df_clean.createOrReplaceTempView("banklist")

df_check = spark.sql("""select `bank_name`, city, `closing_date`, `acquiring_institution` from banklist""")
df_check.show(4, truncate=False)


+-------------------------------------+------------+------------+---------------------------------+
|bank_name                            |city        |closing_date|acquiring_institution            |
+-------------------------------------+------------+------------+---------------------------------+
|The Santa Anna National Bank         |Santa Anna  |27-Jun-25   |Coleman County State Bank        |
|Pulaski Savings Bank                 |Chicago     |17-Jan-25   |Millennium Bank                  |
|First National Bank of Lindsay       |Lindsay     |18-Oct-24   |First Bank & Trust Co.           |
|Republic First Bank dba Republic Bank|Philadelphia|26-Apr-24   |Fulton Bank, National Association|
+-------------------------------------+------------+------------+---------------------------------+
only showing top 4 rows



# Database Basic Operations

In [ ]:
df_clean.describe().show()

+-------+--------------------+-------+-----+------------------+---------------------+------------+------------------+
|summary|           bank_name|   city|state|              cert|acquiring_institution|closing_date|              fund|
+-------+--------------------+-------+-----+------------------+---------------------+------------+------------------+
|  count|                 572|    572|  572|               572|                  572|         572|               572|
|   mean|                NULL|   NULL| NULL|31553.940559440558|                 NULL|        NULL|10044.863636363636|
| stddev|                NULL|   NULL| NULL| 16498.37191914555|                 NULL|        NULL|1108.3189739860254|
|    min|1st American Stat...|Acworth|   AL|             10054|      1st United Bank|    1-Aug-08|             10000|
|    max|               ebank|Wyoming|   WY|              9961|  Your Community Bank|    9-Sep-11|              6006|
+-------+--------------------+-------+-----+------------

In [ ]:
df_clean.describe('city', 'state').show()

+-------+-------+-----+
|summary|   city|state|
+-------+-------+-----+
|  count|    572|  572|
|   mean|   NULL| NULL|
| stddev|   NULL| NULL|
|    min|Acworth|   AL|
|    max|Wyoming|   WY|
+-------+-------+-----+



# Count, Columns and Schema

In [ ]:
print('Total de linhas:', df_clean.count())
print('Total de colunas:', len(df_clean.columns))
print('Colunas:', df_clean.columns)
print('Tipo de dados:', df_clean.dtypes)
print('Schema:', df_clean.schema)


Total de linhas: 572
Total de colunas: 7
Colunas: ['bank_name', 'city', 'state', 'cert', 'acquiring_institution', 'closing_date', 'fund']
Tipo de dados: [('bank_name', 'string'), ('city', 'string'), ('state', 'string'), ('cert', 'string'), ('acquiring_institution', 'string'), ('closing_date', 'string'), ('fund', 'string')]
Schema: StructType([StructField('bank_name', StringType(), True), StructField('city', StringType(), True), StructField('state', StringType(), True), StructField('cert', StringType(), True), StructField('acquiring_institution', StringType(), True), StructField('closing_date', StringType(), True), StructField('fund', StringType(), True)])


In [ ]:
df_clean.printSchema()

root
 |-- bank_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- cert: string (nullable = true)
 |-- acquiring_institution: string (nullable = true)
 |-- closing_date: string (nullable = true)
 |-- fund: string (nullable = true)



# Remove Duplicates

In [ ]:
df_clean = df_clean.dropDuplicates()
print('df.count:', df_clean.count())
print('df.columns:', df_clean.columns)

df.count: 572
df.columns: ['bank_name', 'city', 'state', 'cert', 'acquiring_institution', 'closing_date', 'fund']


# Select Specific Columns

In [ ]:
df_clean2 = df_clean.select(["bank_name", "city"])
df_clean2.show()


+--------------------+---------------+
|           bank_name|           city|
+--------------------+---------------+
|Covenant Bank & T...|    Rock Spring|
|        AmTrust Bank|      Cleveland|
|Chestatee State Bank|    Dawsonville|
|Bank of Florida -...|Fort Lauderdale|
|       Colonial Bank|     Montgomery|
|Imperial Savings ...|   Martinsville|
| First National Bank|       Savannah|
| First Regional Bank|    Los Angeles|
|   The Tattnall Bank|     Reidsville|
|Greater Atlantic ...|         Reston|
|  Park National Bank|        Chicago|
|            BankEast|      Knoxville|
|      Signature Bank|        Windsor|
|Bayside Savings Bank| Port Saint Joe|
|       Frontier Bank|        Everett|
|First Integrity B...|        Staples|
|       Resolute Bank|         Maumee|
|          Excel Bank|        Sedalia|
|The Park Avenue Bank|       New York|
|HarVest Bank of M...|   Gaithersburg|
+--------------------+---------------+
only showing top 20 rows



# Select Miltiple Columns

In [ ]:
col_1 = list(set(df_clean.columns) - {'cert', 'state'})
df_clean2_0 = df_clean.select(*col_1)
df_clean2_0.show()


+---------------+---------------------+-----+--------------------+------------+
|           city|acquiring_institution| fund|           bank_name|closing_date|
+---------------+---------------------+-----+--------------------+------------+
|    Rock Spring|   Stearns Bank, N.A.|10430|Covenant Bank & T...|   23-Mar-12|
|      Cleveland| New York Communit...|10155|        AmTrust Bank|    4-Dec-09|
|    Dawsonville|   Bank of the Ozarks|10320|Chestatee State Bank|   17-Dec-10|
|Fort Lauderdale|             EverBank|10241|Bank of Florida -...|   28-May-10|
|     Montgomery| Branch Banking & ...|10103|       Colonial Bank|   14-Aug-09|
|   Martinsville| River Community B...|10280|Imperial Savings ...|   20-Aug-10|
|       Savannah| The Savannah Bank...|10251| First National Bank|   25-Jun-10|
|    Los Angeles| First-Citizens Ba...|10177| First Regional Bank|   29-Jan-10|
|     Reidsville| Heritage Bank of ...|10153|   The Tattnall Bank|    4-Dec-09|
|         Reston|             Sonabank|1

# Rename Columns

In [ ]:
df_clean2 = df_clean \
    .withColumnRenamed('bank_name', 'bank_name') \
    .withColumnRenamed('acquiring_institution', 'acq_institution') \
    .withColumnRenamed('closing_date', 'closing_date') \
    .withColumnRenamed('state', 'state') \
    .withColumnRenamed('cert', 'cert')  # \
df_clean2.show()


+--------------------+---------------+-----+-----+--------------------+------------+-----+
|           bank_name|           city|state| cert|     acq_institution|closing_date| fund|
+--------------------+---------------+-----+-----+--------------------+------------+-----+
|Covenant Bank & T...|    Rock Spring|   GA|58068|  Stearns Bank, N.A.|   23-Mar-12|10430|
|        AmTrust Bank|      Cleveland|   OH|29776|New York Communit...|    4-Dec-09|10155|
|Chestatee State Bank|    Dawsonville|   GA|34578|  Bank of the Ozarks|   17-Dec-10|10320|
|Bank of Florida -...|Fort Lauderdale|   FL|57360|            EverBank|   28-May-10|10241|
|       Colonial Bank|     Montgomery|   AL| 9609|Branch Banking & ...|   14-Aug-09|10103|
|Imperial Savings ...|   Martinsville|   VA|31623|River Community B...|   20-Aug-10|10280|
| First National Bank|       Savannah|   GA|34152|The Savannah Bank...|   25-Jun-10|10251|
| First Regional Bank|    Los Angeles|   CA|23011|First-Citizens Ba...|   29-Jan-10|10177|

# Add Columns

In [ ]:
df_clean2 = df_clean2.withColumn('ST', col('state'))
df_clean2.show(2)

+--------------------+-----------+-----+-----+--------------------+------------+-----+---+
|           bank_name|       city|state| cert|     acq_institution|closing_date| fund| ST|
+--------------------+-----------+-----+-----+--------------------+------------+-----+---+
|Covenant Bank & T...|Rock Spring|   GA|58068|  Stearns Bank, N.A.|   23-Mar-12|10430| GA|
|        AmTrust Bank|  Cleveland|   OH|29776|New York Communit...|    4-Dec-09|10155| OH|
+--------------------+-----------+-----+-----+--------------------+------------+-----+---+
only showing top 2 rows



# Add constant column

In [ ]:
df_clean2 = df_clean.withColumn('country', lit('US'))
df_clean2.show(2)

+--------------------+-----------+-----+-----+---------------------+------------+-----+-------+
|           bank_name|       city|state| cert|acquiring_institution|closing_date| fund|country|
+--------------------+-----------+-----+-----+---------------------+------------+-----+-------+
|Covenant Bank & T...|Rock Spring|   GA|58068|   Stearns Bank, N.A.|   23-Mar-12|10430|     US|
|        AmTrust Bank|  Cleveland|   OH|29776| New York Communit...|    4-Dec-09|10155|     US|
+--------------------+-----------+-----+-----+---------------------+------------+-----+-------+
only showing top 2 rows



# Drop Columns

In [ ]:
df_clean2 = df_clean.drop('cert')
df_clean2.show(2)

+--------------------+-----------+-----+---------------------+------------+-----+
|           bank_name|       city|state|acquiring_institution|closing_date| fund|
+--------------------+-----------+-----+---------------------+------------+-----+
|Covenant Bank & T...|Rock Spring|   GA|   Stearns Bank, N.A.|   23-Mar-12|10430|
|        AmTrust Bank|  Cleveland|   OH| New York Communit...|    4-Dec-09|10155|
+--------------------+-----------+-----+---------------------+------------+-----+
only showing top 2 rows



# Drop Multiple Columns

In [ ]:
df_clean2 = df_clean.drop('cert', 'state')
df_clean2.show(2)

+--------------------+-----------+---------------------+------------+-----+
|           bank_name|       city|acquiring_institution|closing_date| fund|
+--------------------+-----------+---------------------+------------+-----+
|Covenant Bank & T...|Rock Spring|   Stearns Bank, N.A.|   23-Mar-12|10430|
|        AmTrust Bank|  Cleveland| New York Communit...|    4-Dec-09|10155|
+--------------------+-----------+---------------------+------------+-----+
only showing top 2 rows



In [ ]:
df_clean2 = reduce(DataFrame.drop, ['cert', 'state'], df_clean)
df_clean2.show(2)

+--------------------+-----------+---------------------+------------+-----+
|           bank_name|       city|acquiring_institution|closing_date| fund|
+--------------------+-----------+---------------------+------------+-----+
|Covenant Bank & T...|Rock Spring|   Stearns Bank, N.A.|   23-Mar-12|10430|
|        AmTrust Bank|  Cleveland| New York Communit...|    4-Dec-09|10155|
+--------------------+-----------+---------------------+------------+-----+
only showing top 2 rows



# Filter Data

In [ ]:
print('df.columns:', df_clean.columns)

df.columns: ['bank_name', 'city', 'state', 'cert', 'acquiring_institution', 'closing_date', 'fund']


In [ ]:
# Equal to values
df_clean2 = df_clean.where(df_clean['state'] == 'NE')

# Between values
df_clean3 = df_clean.where(df_clean['cert'].between('1000', '2000'))

# Is in multiple values
df_clean4 = df_clean.where(df_clean['state'].isin('NE', 'IL'))

print('df_clean.count :', df_clean.count())
print('df_clean2.count:', df_clean2.count())
print('df_clean3.count:', df_clean3.count())
print('df4_clean.count:', df_clean4.count())


df_clean.count : 572
df_clean2.count: 4
df_clean3.count: 97
df4_clean.count: 74


# Filter Data Using Logical Operators

In [ ]:
df_clean2 = df_clean.where((df_clean['state'] == 'NE') & (df_clean['city'] == 'Ericson'))
df_clean2.show(3)


+------------------+-------+-----+-----+---------------------+------------+-----+
|         bank_name|   city|state| cert|acquiring_institution|closing_date| fund|
+------------------+-------+-----+-----+---------------------+------------+-----+
|Ericson State Bank|Ericson|   NE|18265| Farmers and Merch...|   14-Feb-20|10535|
+------------------+-------+-----+-----+---------------------+------------+-----+



# Replace(Substituir) Values in Dataframe

In [ ]:
# Pre replace
df_clean.show(2)

# Post replace
print('Replace 7 in the above dataframe with 17 at all instances')
df_clean.na.replace(7, 17).show(2)


+--------------------+-----------+-----+-----+---------------------+------------+-----+
|           bank_name|       city|state| cert|acquiring_institution|closing_date| fund|
+--------------------+-----------+-----+-----+---------------------+------------+-----+
|Covenant Bank & T...|Rock Spring|   GA|58068|   Stearns Bank, N.A.|   23-Mar-12|10430|
|        AmTrust Bank|  Cleveland|   OH|29776| New York Communit...|    4-Dec-09|10155|
+--------------------+-----------+-----+-----+---------------------+------------+-----+
only showing top 2 rows

Replace 7 in the above dataframe with 17 at all instances
+--------------------+-----------+-----+-----+---------------------+------------+-----+
|           bank_name|       city|state| cert|acquiring_institution|closing_date| fund|
+--------------------+-----------+-----+-----+---------------------+------------+-----+
|Covenant Bank & T...|Rock Spring|   GA|58068|   Stearns Bank, N.A.|   23-Mar-12|10430|
|        AmTrust Bank|  Cleveland|   